In [13]:
!pip install dotenv

In [14]:
import os
import time
import smtplib
import pandas as pd
from pathlib import Path
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from dotenv import load_dotenv

load_dotenv()

# --- CONFIGURATION ---
SMTP_SERVER = os.getenv('SMTP_SERVER')
SMTP_PORT = os.getenv('SMTP_PORT')
SMTP_USERNAME = os.getenv('SMTP_USERNAME')
SMTP_PASSWORD = os.getenv('SMTP_PASSWORD')
SENDER_EMAIL = os.getenv('SENDER_EMAIL')
SENDER_NAME = os.getenv('SENDER_NAME')

config = {
    'SMTP_SERVER': SMTP_SERVER,
    'SMTP_PORT': SMTP_PORT,
    'SMTP_USERNAME': SMTP_USERNAME,
    'SMTP_PASSWORD': SMTP_PASSWORD,
    'SENDER_EMAIL': SENDER_EMAIL,
    'SENDER_NAME': SENDER_NAME
}

for name, value in config.items():
    if value is None:
        print(f"Error loading field: {name}")   

SHEET_FILENAME = "guests_data.csv"
QR_FOLDER = "QR code data"

# --- BATCH CONTROL ---
BATCH_SIZE = 50
DELAY_BETWEEN_EMAILS = 1.0

# --- EMAIL TEMPLATE ---
EMAIL_SUBJECT = "Your Official Ticket & QR Code for the Event"

EMAIL_TEMPLATE_HTML = """
<!DOCTYPE html>
<html>
<body style="font-family: Arial, sans-serif; color: #333; line-height: 1.6; margin: 0; padding: 0;">
    <div style="max-width: 600px; margin: 20px auto; padding: 20px; border: 1px solid #e0e0e0; border-radius: 10px; background-color: #ffffff;">
        <h2 style="color: #007aff; margin-top: 0;">Hello {name},</h2>
        <p>We are excited to welcome you to the upcoming event!</p>
        <p>Your registration is confirmed. Please keep this email safe and present the QR code below at the gate for entry verification.</p>
        
        <div style="text-align: center; margin: 30px 0; padding: 15px; background: #f9f9f9; border-radius: 8px;">
            <img src="cid:qrcode_img" alt="Your QR Code" style="width: 220px; height: 220px; border: 2px solid #ddd; padding: 5px; background: #fff; border-radius: 8px;">
            <p style="font-size: 0.85rem; color: #666; margin-top: 8px; font-family: monospace;">Ticket ID: {uuid}</p>
        </div>
        
        <p>If you have any issues or questions, reply directly to this email.</p>
        <br>
        <p>Best regards,<br><strong>{sender_name}</strong></p>
    </div>
</body>
</html>
"""

In [15]:
local_file = Path(SHEET_FILENAME)
if not local_file.exists():
    raise FileNotFoundError(f"Cannot find '{SHEET_FILENAME}'. Run the QR generation notebook first.")

df = pd.read_csv(local_file)

# Ensure 'email_sent' column exists for state tracking
if 'email_sent' not in df.columns:
    df['email_sent'] = False

# Normalize data type to boolean
df['email_sent'] = df['email_sent'].fillna(False).astype(bool)

unsent_count = len(df[~df['email_sent']])
sent_count = len(df[df['email_sent']])

print(f"📊 Total Attendees: {len(df)}")
print(f"✅ Emails Already Sent: {sent_count}")
print(f"⏳ Remaining to Send: {unsent_count}")

📊 Total Attendees: 1
✅ Emails Already Sent: 0
⏳ Remaining to Send: 1


In [16]:
def send_qr_email(smtp_conn, recipient_email, recipient_name, uuid_val):
    qr_file_path = Path(QR_FOLDER) / f"{uuid_val}.png"
    
    if not qr_file_path.exists():
        print(f"⚠️ Missing QR code file for {recipient_name} ({uuid_val}). Skipping.")
        return False

    msg = MIMEMultipart('related')
    msg['Subject'] = EMAIL_SUBJECT
    msg['From'] = f"{SENDER_NAME} <{SENDER_EMAIL}>"
    msg['To'] = recipient_email

    # Inject dynamic fields into the HTML template
    html_content = EMAIL_TEMPLATE_HTML.format(
        name=recipient_name,
        uuid=uuid_val,
        sender_name=SENDER_NAME
    )
    
    msg_html = MIMEText(html_content, 'html')
    msg.attach(msg_html)

    # Embed QR Code as an inline image (cid:qrcode_img)
    with open(qr_file_path, 'rb') as f:
        img_data = f.read()
        img = MIMEImage(img_data)
        img.add_header('Content-ID', '<qrcode_img>')
        img.add_header('Content-Disposition', 'inline', filename=f"{uuid_val}.png")
        msg.attach(img)

    smtp_conn.send_message(msg)
    return True

In [18]:
pending_df = df[~df['email_sent']].head(BATCH_SIZE)

if pending_df.empty:
    print("🎉 All emails have been sent! Nothing left to process.")
else:
    print(f"🚀 Starting dispatch for batch of {len(pending_df)} emails...\n")
    
    try:
        # Establish SMTP Connection
        if SMTP_PORT == 465:
            server = smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT)
        else:
            server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
            server.starttls()
            
        server.login(SMTP_USERNAME, SMTP_PASSWORD)
        
        sent_in_this_batch = 0

        for idx, row in pending_df.iterrows():
            guest_name = str(row['name'])
            guest_email = str(row['email']).strip()
            guest_uuid = str(row['uuid']).strip()

            try:
                success = send_qr_email(server, guest_email, guest_name, guest_uuid)
                if success:
                    # Update state in memory
                    df.at[idx, 'email_sent'] = True
                    sent_in_this_batch += 1
                    print(f"✅ ({sent_in_this_batch}/{len(pending_df)}) Sent to {guest_name} <{guest_email}>")
                    
                    # Instantly save to disk to survive crashes/cancellations
                    df.to_csv(SHEET_FILENAME, index=False)
                    
                time.sleep(DELAY_BETWEEN_EMAILS)

            except Exception as e:
                print(f"❌ Failed to send to {guest_name} ({guest_email}): {e}")

        server.quit()
        print(f"\n✨ Batch complete! Successfully sent {sent_in_this_batch} emails. State saved to {SHEET_FILENAME}.")

    except Exception as e:
        print(f"❌ SMTP Connection Error: {e}")

🎉 All emails have been sent! Nothing left to process.
